# 🧹Clean HTE
The purpose of this notebook is to clean the excel files.

In [65]:
import pandas as pd
from datetime import datetime
data = pd.ExcelFile("./datasets/hte-2021-2026.xlsx")

In [77]:
def clean_df(path, sheet_name):
    data =  pd.ExcelFile(path)
    df = data.parse(sheet_name=sheet_name, skiprows=2, ignore_index=True)
    df = df.iloc[:, 1:]
    df.columns = [col for col in df.columns.str.strip().str.lower().str.replace(r'[\'-]', '', regex=True).str.replace(r'[\s+()/]', '_', regex=True)]
    df = df.dropna(subset=['expiry_date', 'email_address'])
    # df_2026[['duration', 'duration_unit']] = df_2026['validity'].str.split(expand=True)
    df['contact_person'] = df['contact_person'].str.title()
    df['expiry_year'] = df['expiry_date'].apply(pd.to_datetime, errors='coerce').dt.year
    df['is_valid'] = df['expiry_year'].apply(lambda x: True if x >= datetime.now().year else False)
    df = df[[
        'companys_name',
        'contact_person',
        'position',
        'email_address',
        'address',
        'expiry_date',
        'expiry_year',
        'is_valid'
    ]]
    df = df[df['is_valid'] == True]
    return df


In [81]:
df_2026 = clean_df("./datasets/hte-2021-2026.xlsx", "2026")
df_2025 = clean_df("./datasets/hte-2021-2026.xlsx", "2025")

df_202526 = pd.concat([df_2026, df_2025], ignore_index=True)